In [0]:
import pyspark.pandas as ps
import requests

In [0]:
res = requests.get("https://api.census.gov/data/2024/acs/acs5/subject?get=group(S1903)&ucgid=0100000US")

In [0]:
import requests, json

# 1️⃣ Get the variable dictionary
meta_url = "https://api.census.gov/data/2019/acs/acs1/variables.json"
meta = requests.get(meta_url).json()["variables"]

# 2️⃣ Build a simple lookup: code → label
lookup = {code: info["label"] for code, info in meta.items()}

# 3️⃣ Fetch the data you need
data_url = "https://api.census.gov/data/2024/acs/acs5/subject?get=group(S1903)&ucgid=0100000US"
rows = requests.get(data_url).json()

# 4️⃣ Replace the header row with plain English labels
header = rows[0]
plain_header = [lookup.get(col, col) for col in header]  # fallback to original if unknown
rows[0] = plain_header

# Wrap each row as a dict for schema inference, ensure all values are strings
data_list = [dict(zip(plain_header, [str(val) if val is not None else '' for val in row])) for row in rows[1:]]

df_census = spark.createDataFrame(data_list)
display(df_census)